## Ejercicio 1: Evaluación de solicitud de crédito bancario

**Curso:** Inteligencia Artificial I (IS-484) · **Alumno:** Henry Josue Flores Saras

### Ficha PEAS

| Componente | Descripción |
|---|---|
| **Percepción (S)** | `ingreso_mensual` (soles), `monto_solicitado` (soles), `tiene_historial_moroso` (True/False). A partir de ellos el agente calcula `relacion = monto_solicitado / ingreso_mensual`. |
| **Acciones (A)** | `aprobar`, `aprobar con condiciones` (garantía, aval o plazo/monto ajustado), `rechazar`. Cada acción va acompañada de un motivo en texto. |
| **Entorno (E)** | Sistema de evaluación crediticia del banco: solicitudes que llegan de clientes con distintos ingresos y comportamientos de pago, políticas internas de riesgo y regulación de la SBS. |
| **Objetivo** | Otorgar crédito a quienes pueden pagarlo y evitar préstamos con alta probabilidad de impago. |
| **Medida de desempeño (P)** | Tasa de morosidad de los créditos aprobados (baja), proporción de buenos clientes aprobados (alta), coherencia de la decisión con la política de riesgo y tiempo de respuesta. |

### Justificación de las reglas

La relación monto/ingreso indica cuántos meses de ingreso representa la deuda: si es ≤ 3, la cuota mensual en un plazo típico de 12 a 24 meses queda por debajo del ~25 % del ingreso, por lo que es pagable. Entre 3 y 6 el riesgo es moderado y el banco lo mitiga pidiendo garantía o aval; por encima de 6 la carga es excesiva. El historial moroso es la mejor señal de impago futuro, por eso a esos clientes nunca se les aprueba directamente: solo se les da crédito condicionado si el monto es pequeño (relación ≤ 1) y se rechaza en cualquier otro caso. Un ingreso no positivo se rechaza porque no hay capacidad de pago que evaluar.

### Código

In [4]:
UMBRAL_APROBAR = 3.0         # relacion <= 3 -> aprobar (sin historial moroso)
UMBRAL_CONDICIONES = 6.0     # 3 < relacion <= 6 -> aprobar con condiciones
UMBRAL_MOROSO = 1.0          # con historial moroso: relacion <= 1 -> condiciones


def agente_credito(ingreso_mensual, monto_solicitado, tiene_historial_moroso):
    """Agente reactivo simple: percibe la solicitud y retorna (accion, motivo)."""
    # Validacion de percepciones (evita divisiones por cero y datos corruptos)
    if ingreso_mensual <= 0:
        return "rechazar", "ingreso mensual no valido (menor o igual a 0), no hay capacidad de pago"
    if monto_solicitado <= 0:
        return "rechazar", "monto solicitado no valido (menor o igual a 0)"

    relacion = monto_solicitado / ingreso_mensual

    if tiene_historial_moroso:
        if relacion <= UMBRAL_MOROSO:
            return ("aprobar con condiciones",
                    f"historial moroso, pero relacion monto/ingreso baja ({relacion:.2f} <= {UMBRAL_MOROSO}); se exige aval o garantia")
        return ("rechazar",
                f"historial moroso y relacion monto/ingreso de {relacion:.2f} mayor a {UMBRAL_MOROSO}")

    if relacion <= UMBRAL_APROBAR:
        return ("aprobar",
                f"sin historial moroso y relacion monto/ingreso de {relacion:.2f} <= {UMBRAL_APROBAR}")
    if relacion <= UMBRAL_CONDICIONES:
        return ("aprobar con condiciones",
                f"sin historial moroso, pero relacion de {relacion:.2f} entre {UMBRAL_APROBAR} y {UMBRAL_CONDICIONES}; se exige garantia o menor plazo")
    return ("rechazar",
            f"relacion monto/ingreso de {relacion:.2f} mayor a {UMBRAL_CONDICIONES}, cuota excesiva para el ingreso")


### Simulación y pruebas

In [ ]:

casos = [
    # (ingreso, monto, moroso, descripcion)
    (3000, 5000,  False, "relacion baja, sin moroso"),
    (2000, 6000,  False, "BORDE: relacion exactamente 3.0"),
    (2000, 6020,  False, "BORDE: relacion apenas mayor a 3.0 (3.01)"),
    (1500, 7500,  False, "relacion media (5.0), sin moroso"),
    (1200, 12000, False, "relacion alta (10.0), sin moroso"),
    (4000, 4000,  True,  "BORDE: moroso con relacion exactamente 1.0"),
    (2500, 5000,  True,  "moroso con relacion 2.0"),
    (0,    3000,  False, "dato invalido: ingreso 0"),
]

print(f"{'Ingreso':>8} {'Monto':>8} {'Moroso':>7} {'Rel.':>6} | {'Accion':<24} | Motivo")
print("-" * 120)
for ingreso, monto, moroso, desc in casos:
    accion, motivo = agente_credito(ingreso, monto, moroso)
    rel = monto / ingreso if ingreso > 0 else float("nan")
    print(f"{ingreso:>8} {monto:>8} {str(moroso):>7} {rel:>6.2f} | {accion:<24} | {motivo}")

 Ingreso    Monto  Moroso   Rel. | Accion                   | Motivo
------------------------------------------------------------------------------------------------------------------------
    3000     5000   False   1.67 | aprobar                  | sin historial moroso y relacion monto/ingreso de 1.67 <= 3.0
    2000     6000   False   3.00 | aprobar                  | sin historial moroso y relacion monto/ingreso de 3.00 <= 3.0
    2000     6020   False   3.01 | aprobar con condiciones  | sin historial moroso, pero relacion de 3.01 entre 3.0 y 6.0; se exige garantia o menor plazo
    1500     7500   False   5.00 | aprobar con condiciones  | sin historial moroso, pero relacion de 5.00 entre 3.0 y 6.0; se exige garantia o menor plazo
    1200    12000   False  10.00 | rechazar                 | relacion monto/ingreso de 10.00 mayor a 6.0, cuota excesiva para el ingreso
    4000     4000    True   1.00 | aprobar con condiciones  | historial moroso, pero relacion monto/ingreso baja (1.

### Visualización